### Tumor Segmentation Notebook

#### Load Librairies

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from tqdm import tqdm

from MobileUnetR.mobile_unetr import MobileUNETR
from segmentation_dataset import make_plane_dataloaders, SEGMENTATION_CLASSES

#### Defining Training Hyperparameters 

In [ ]:
# Data and training configuration
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'datasets' / 'brisc2025').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_ROOT = PROJECT_ROOT / 'datasets' / 'brisc2025'
CHECKPOINT_ROOT = PROJECT_ROOT / 'MobileUnetR' / 'checkpoints'

BATCH_SIZE = 8
LEARNING_RATE = 1e-4
EPOCHS = 100
NUM_WORKERS = 0
IMAGE_SIZE = (256, 256)
NUM_CLASSES = 4
PLANES = ('axial', 'coronal', 'sagittal')

if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")

print(f"Using device: {DEVICE}")


#### Dataset Preparation

In [ ]:
class CustomLoss(nn.Module):
    def __init__(self, ce_weight=2.0, tversky_weight=1.0, alpha=0.3, beta=0.7, gamma=2.0, eps=1e-6):
        super().__init__()
        self.ce_weight = ce_weight
        self.tversky_weight = tversky_weight
        self.alpha = alpha
        self.beta = beta
        self.gamma = gamma
        self.eps = eps
        self.cross_entropy = nn.CrossEntropyLoss()

    def forward(self, logits, target):
        if logits.ndim != 4 or logits.shape[1] != NUM_CLASSES:
            raise ValueError(
                f"Expected logits [N, {NUM_CLASSES}, H, W], got {tuple(logits.shape)}"
            )
        if target.ndim == 4 and target.shape[1] == 1:
            target = target.squeeze(1)
        if tuple(target.shape) != (logits.shape[0], *logits.shape[2:]):
            raise ValueError(
                f"Expected target [N, H, W], got {tuple(target.shape)}"
            )

        target = target.long()
        if target.numel() and (target.min() < 0 or target.max() >= NUM_CLASSES):
            raise ValueError(f"Target labels must be between 0 and {NUM_CLASSES - 1}")

        base_loss = self.cross_entropy(logits, target)
        probs = torch.softmax(logits, dim=1)
        target_one_hot = torch.nn.functional.one_hot(
            target, num_classes=NUM_CLASSES
        ).movedim(-1, 1).to(dtype=probs.dtype)
        dims = (0, 2, 3)
        tp = (probs * target_one_hot).sum(dim=dims)
        fp = (probs * (1.0 - target_one_hot)).sum(dim=dims)
        fn = ((1.0 - probs) * target_one_hot).sum(dim=dims)

        tversky = (tp + self.eps) / (
            tp + self.alpha * fp + self.beta * fn + self.eps
        )
        tversky_loss = (1.0 - tversky.mean()).clamp_min(0.0).pow(1.0 / self.gamma)
        return self.ce_weight * base_loss + self.tversky_weight * tversky_loss


def compute_iou_per_class(pred_logits, target):
    pred_mask = pred_logits.argmax(dim=1)
    if target.ndim == 4 and target.shape[1] == 1:
        target = target.squeeze(1)
    metrics = {}

    for class_index, class_name in enumerate(SEGMENTATION_CLASSES):
        pred_class = pred_mask == class_index
        target_class = target == class_index
        intersection = (pred_class & target_class).sum().item()
        union = (pred_class | target_class).sum().item()
        metrics[class_name] = intersection / union if union else 1.0

    return metrics

models = nn.ModuleDict({
    plane: MobileUNETR(in_channels=3, num_classes=NUM_CLASSES).to(DEVICE)
    for plane in PLANES
})
optimizers = {
    plane: torch.optim.AdamW(models[plane].parameters(), lr=LEARNING_RATE)
    for plane in PLANES
}
criterion = CustomLoss().to(DEVICE)

print(models['axial'])


In [ ]:

from torch.utils.data import random_split
loaders = make_plane_dataloaders(
    DATA_ROOT,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    image_size=IMAGE_SIZE,
)

for plane in PLANES:
    full_train_dataset = loaders[plane]['train'].dataset
    val_count = max(1, round(0.1 * len(full_train_dataset)))
    train_count = len(full_train_dataset) - val_count
    train_dataset, val_dataset = random_split(
        full_train_dataset,
        (train_count, val_count),
        generator=torch.Generator().manual_seed(42),
    )
    loaders[plane]['train'] = DataLoader(
        train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS
    )
    loaders[plane]['val'] = DataLoader(
        val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS
    )
    train_images, train_masks = next(iter(loaders[plane]['train']))
    print(
        f"{plane}: train={len(loaders[plane]['train'].dataset)}, "
        f"val={len(loaders[plane]['val'].dataset)}, "
        f"test={len(loaders[plane]['test'].dataset)}, "
        f"images={tuple(train_images.shape)}, masks={tuple(train_masks.shape)}, "
        f"labels={torch.unique(train_masks).tolist()}"
    )


#### Training loop

In [ ]:
def run_epoch(model, loader, device, optimizer=None, log_every=10):
    is_training = optimizer is not None
    model.train(is_training)
    running_loss = 0.0
    running_iou = {name: 0.0 for name in SEGMENTATION_CLASSES}
    total_seen = 0

    with torch.set_grad_enabled(is_training):
        for batch_idx, (images, targets) in enumerate(loader, start=1):
            images = images.to(device, dtype=torch.float32)
            targets = targets.to(device, dtype=torch.long)

            if is_training:
                optimizer.zero_grad()
            logits = model(images)
            loss = criterion(logits, targets)
            if is_training:
                loss.backward()
                optimizer.step()

            batch_size = images.size(0)
            running_loss += loss.item() * batch_size
            total_seen += batch_size
            batch_iou = compute_iou_per_class(logits.detach(), targets)
            for name in running_iou:
                running_iou[name] += batch_iou[name] * batch_size

            if is_training and (batch_idx % log_every == 0 or batch_idx == len(loader)):
                mean_iou = sum(running_iou.values()) / (len(running_iou) * total_seen)
                print(
                    f"  Batch {batch_idx}/{len(loader)} - "
                    f"loss={running_loss / total_seen:.4f} - mIoU={mean_iou:.4f}"
                )

    if total_seen == 0:
        raise ValueError("Cannot compute metrics for an empty data loader")
    mean_iou = {name: value / total_seen for name, value in running_iou.items()}
    return running_loss / total_seen, mean_iou


history = {
    plane: {"train_loss": [], "val_loss": [], "train_iou": [], "val_iou": []}
    for plane in PLANES
}
best_val_loss = {plane: float('inf') for plane in PLANES}

for epoch in tqdm(range(EPOCHS)):
    for plane in PLANES:
        print(f"Epoch {epoch + 1}/{EPOCHS} - {plane}")
        train_loss, train_iou = run_epoch(
            models[plane], loaders[plane]['train'], DEVICE,
            optimizer=optimizers[plane], log_every=10,
        )
        val_loss, val_iou = run_epoch(
            models[plane], loaders[plane]['val'], DEVICE
        )

        plane_history = history[plane]
        plane_history['train_loss'].append(train_loss)
        plane_history['val_loss'].append(val_loss)
        plane_history['train_iou'].append(train_iou)
        plane_history['val_iou'].append(val_iou)
        print(
            f"  train_loss={train_loss:.4f} val_loss={val_loss:.4f} "
            f"train_mIoU={sum(train_iou.values()) / NUM_CLASSES:.4f} "
            f"val_mIoU={sum(val_iou.values()) / NUM_CLASSES:.4f}"
        )

        checkpoint_dir = CHECKPOINT_ROOT / plane
        checkpoint_dir.mkdir(parents=True, exist_ok=True)
        checkpoint = {
            "epoch": epoch + 1,
            "plane": plane,
            "class_names": SEGMENTATION_CLASSES,
            "model_state_dict": models[plane].state_dict(),
            "optimizer_state_dict": optimizers[plane].state_dict(),
            "train_loss": train_loss,
            "val_loss": val_loss,
            "val_iou": val_iou,
        }
        torch.save(checkpoint, checkpoint_dir / 'current_model.pt')
        if val_loss < best_val_loss[plane]:
            best_val_loss[plane] = val_loss
            torch.save(checkpoint, checkpoint_dir / 'best_model.pt')

for plane in PLANES:
    checkpoint_path = CHECKPOINT_ROOT / plane / 'best_model.pt'
    checkpoint = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)
    models[plane].load_state_dict(checkpoint['model_state_dict'])
    test_loss, test_iou = run_epoch(models[plane], loaders[plane]['test'], DEVICE)
    history[plane]['test_loss'] = test_loss
    history[plane]['test_iou'] = test_iou
    print(
        f"{plane} test: loss={test_loss:.4f}, "
        f"mIoU={sum(test_iou.values()) / NUM_CLASSES:.4f}, IoU={test_iou}"
    )

print('Training finished.')
print(history)


In [ ]:
PLANE_TO_INSPECT = 'axial'
checkpoint_path = CHECKPOINT_ROOT / PLANE_TO_INSPECT / 'best_model.pt'
if not checkpoint_path.is_file():
    raise FileNotFoundError(
        f"No trained checkpoint found at {checkpoint_path}. Run the training cell first."
    )

checkpoint = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)
inference_model = MobileUNETR(in_channels=3, num_classes=NUM_CLASSES).to(DEVICE)
inference_model.load_state_dict(checkpoint['model_state_dict'])
inference_model.eval()
print(f"Loaded {PLANE_TO_INSPECT} checkpoint from epoch {checkpoint['epoch']}")

test_dataset = loaders[PLANE_TO_INSPECT]['test'].dataset
sample_count = min(6, len(test_dataset))
figure, axes = plt.subplots(sample_count, 3, figsize=(12, 4 * sample_count), squeeze=False)
with torch.no_grad():
    for row, index in enumerate(range(sample_count)):
        image, target = test_dataset[index]
        logits = inference_model(image.unsqueeze(0).to(DEVICE))
        prediction = logits.argmax(dim=1).squeeze(0).cpu()

        axes[row, 0].imshow(image.permute(1, 2, 0).cpu().numpy())
        axes[row, 0].set_title('Image')
        axes[row, 1].imshow(target.cpu().numpy(), vmin=0, vmax=NUM_CLASSES - 1)
        axes[row, 1].set_title('Ground truth classes')
        axes[row, 2].imshow(prediction.numpy(), vmin=0, vmax=NUM_CLASSES - 1)
        axes[row, 2].set_title('Predicted classes')
        for column in range(3):
            axes[row, column].axis('off')

plt.tight_layout()
plt.show()


### Export to ONNX